In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
s_trans_df = spark.read.table("`jarvis-catalog`.silver.transactions_data")
s_cards_df = spark.read.table("`jarvis-catalog`.silver.cards_data")
s_users_df = spark.read.table("`jarvis-catalog`.silver.users_data")

In [0]:
#Which day(s) of the week sees the highest number of fraudulent transactions?
fraud_by_day = (
    s_trans_df
    .filter(F.col("fraud") == "Yes")
    .withColumn("day_of_week", F.date_format("date", "EEEE"))
    .groupBy("day_of_week")
    .count()
    .orderBy(F.desc("count"))
)
fraud_by_day.write.mode("overwrite").saveAsTable("`jarvis-catalog`.gold.fraud_by_day")
fraud_by_day.show()

+-----------+-----+
|day_of_week|count|
+-----------+-----+
|     Sunday| 2646|
|     Friday| 2284|
|   Thursday| 2082|
|    Tuesday| 2037|
|     Monday| 1747|
|   Saturday| 1434|
|  Wednesday| 1102|
+-----------+-----+



Fraud Rate trend (fraudulent transactions divided by total) over the past month

In [0]:
fraud_weekly_trend = (
    s_trans_df
    .withColumn(
        "month",
        F.date_trunc("month", "date")
    )
    .groupBy("month")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum(F.when(F.col("fraud") == "Yes", 1).otherwise(0)).alias("fraudulent_transactions")
    )
    .withColumn(
        "fraud_rate",
        F.round(
            (F.col("fraudulent_transactions") / F.col("total_transactions")) * 100,
            2
        )
    )
    .orderBy("month")
)

fraud_weekly_trend.write.mode("overwrite").saveAsTable("`jarvis-catalog`.gold.fraud_weekly_trend")
fraud_weekly_trend.show()

+-------------------+------------------+-----------------------+----------+
|              month|total_transactions|fraudulent_transactions|fraud_rate|
+-------------------+------------------+-----------------------+----------+
|2010-01-01 00:00:00|            101209|                    107|      0.11|
|2010-02-01 00:00:00|             93470|                    259|      0.28|
|2010-03-01 00:00:00|            103345|                    261|      0.25|
|2010-04-01 00:00:00|            100169|                    237|      0.24|
|2010-05-01 00:00:00|            104773|                    274|      0.26|
|2010-06-01 00:00:00|            102677|                    182|      0.18|
|2010-07-01 00:00:00|            106034|                    244|      0.23|
|2010-08-01 00:00:00|            107547|                    229|      0.21|
|2010-09-01 00:00:00|            103902|                    193|      0.19|
|2010-10-01 00:00:00|            106150|                    224|      0.21|
|2010-11-01 

Which users have the largest number of flagged (“is_fraud = true”) transactions?

In [0]:
top_fraud_users = (
    s_trans_df
    .filter(F.col("fraud") == "Yes")
    .groupBy("client_id")
    .agg(
        F.count("*").alias("fraud_transaction_count")
    )
    .orderBy(F.desc("fraud_transaction_count"))
)

top_fraud_users.write.mode("overwrite").saveAsTable("`jarvis-catalog`.gold.top_fraud_users")
top_fraud_users.show()

+---------+-----------------------+
|client_id|fraud_transaction_count|
+---------+-----------------------+
|     1102|                     58|
|      209|                     52|
|       27|                     45|
|      155|                     44|
|     1128|                     43|
|     1741|                     42|
|     1851|                     42|
|      989|                     42|
|     1649|                     41|
|      408|                     39|
|     1926|                     39|
|      692|                     39|
|     1416|                     39|
|      359|                     39|
|     1571|                     39|
|     1992|                     38|
|     1725|                     38|
|     1341|                     37|
|     1569|                     37|
|      561|                     36|
+---------+-----------------------+
only showing top 20 rows


Are there any users showing a sharp rise in transaction amount compared to their weekly average?

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Step 1: Aggregate transaction amount by user and week
weekly_transactions = (
    s_trans_df
    .withColumn("week", F.date_trunc("week", "date"))
    .groupBy("client_id", "week")
    .agg(
        F.sum("amount").alias("weekly_amount")
    )
)

# Step 2: Calculate user's average weekly transaction amount
user_avg_window = Window.partitionBy("client_id")

weekly_trend = (
    weekly_transactions
    .withColumn(
        "avg_weekly_amount",
        F.avg("weekly_amount").over(user_avg_window)
    )
    .withColumn(
        "increase_percentage",
        F.round(
            (
                (F.col("weekly_amount") - F.col("avg_weekly_amount"))
                / F.col("avg_weekly_amount")
            ) * 100,
            2
        )
    )
)

# Step 3: Find sharp increases (example: >100% above average)
sharp_rises = (
    weekly_trend
    .filter(F.col("increase_percentage") > 100)
    .orderBy(F.desc("increase_percentage"))
)

sharp_rises.write.mode("overwrite").saveAsTable("`jarvis-catalog`.gold.sharp_rises")
sharp_rises.show()

+---------+-------------------+------------------+------------------+-------------------+
|client_id|               week|     weekly_amount| avg_weekly_amount|increase_percentage|
+---------+-------------------+------------------+------------------+-------------------+
|     1364|2015-05-04 00:00:00|           2582.34|111.81741245136186|            2209.43|
|     1016|2013-03-04 00:00:00|           1925.35|102.72303921568621|            1774.31|
|      608|2015-12-14 00:00:00|           3627.24|253.32022075055178|            1331.88|
|     1942|2018-12-31 00:00:00|           1392.79| 103.1839105058365|            1249.81|
|     1649|2014-02-03 00:00:00|           1768.95|159.43464566929134|            1009.51|
|     1805|2010-12-13 00:00:00|           4272.23|401.69764591439696|             963.54|
|     1530|2014-02-10 00:00:00|           1252.19|119.88833984375006|             944.46|
|     1998|2015-10-26 00:00:00|1463.0199999999998|145.72216374268996|             903.98|
|      331

Which merchant categories exhibit the highest fraud rate?


In [0]:
merchant_fraud_rate = (
    s_trans_df
    .groupBy("merchant_category")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum(
            F.when(F.col("fraud") == "Yes", 1).otherwise(0)
        ).alias("fraudulent_transactions")
    )
    .withColumn(
        "fraud_rate",
        F.round(
            (F.col("fraudulent_transactions") / F.col("total_transactions")) * 100,
            2
        )
    )
    .orderBy(F.desc("fraud_rate"))
)

merchant_fraud_rate.write.mode("overwrite").saveAsTable("`jarvis-catalog`.gold.merchant_fraud_rate")
merchant_fraud_rate.show()

+--------------------+------------------+-----------------------+----------+
|   merchant_category|total_transactions|fraudulent_transactions|fraud_rate|
+--------------------+------------------+-----------------------+----------+
|        Cruise Lines|               428|                    165|     38.55|
|Music Stores - Mu...|               319|                     76|     23.82|
|Miscellaneous Fab...|               351|                     29|      8.26|
|Computers, Comput...|              2793|                    204|       7.3|
|Floor Covering St...|               334|                     23|      6.89|
|  Electronics Stores|              6997|                    402|      5.75|
|Miscellaneous Met...|               391|                     22|      5.63|
|Fabricated Struct...|               408|                     22|      5.39|
|Precious Stones a...|              5180|                    242|      4.67|
|Coated and Lamina...|               381|                     17|      4.46|

How does fraud distribution vary by time of day (morning vs night)?

In [0]:
fraud_time_analysis = (
    s_trans_df
    .groupBy("time_period")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum(
            F.when(F.col("fraud") == "Yes", 1).otherwise(0)
        ).alias("fraud_transactions")
    )
    .withColumn(
        "fraud_rate",
        F.round(
            (F.col("fraud_transactions") / F.col("total_transactions")) * 100,
            2
        )
    )
    .orderBy(F.desc("fraud_rate"))
)

fraud_time_analysis.write.mode("overwrite").saveAsTable("`jarvis-catalog`.gold.fraud_time_analysis")
fraud_time_analysis.show()

+-----------+------------------+------------------+----------+
|time_period|total_transactions|fraud_transactions|fraud_rate|
+-----------+------------------+------------------+----------+
|         PM|           7302813|              7252|       0.1|
|         AM|           6003102|              6080|       0.1|
+-----------+------------------+------------------+----------+



What’s the average transaction amount for fraud vs non-fraud transactions?

In [0]:
avg_transaction_fraud = (
    s_trans_df
    .groupBy("fraud")
    .agg(
        F.round(
            F.avg("amount"),
            2
        ).alias("average_transaction_amount"),
        F.count("*").alias("transaction_count")
    )
    .orderBy("fraud")
)

avg_transaction_fraud.write.mode("overwrite").saveAsTable("`jarvis-catalog`.gold.avg_transaction_fraud")
avg_transaction_fraud.show()

+-------+--------------------------+-----------------+
|  fraud|average_transaction_amount|transaction_count|
+-------+--------------------------+-----------------+
|     No|                     42.85|          8901631|
|Unknown|                     43.03|          4390952|
|    Yes|                    110.23|            13332|
+-------+--------------------------+-----------------+



Which merchant category has the highest total fraud amount?


In [0]:
merchant_fraud_amount = (
    s_trans_df
    .filter(F.col("fraud") == "Yes")
    .groupBy("merchant_category")
    .agg(
        F.sum("amount").alias("total_fraud_amount"),
        F.count("*").alias("fraud_transaction_count")
    )
    .orderBy(F.desc("total_fraud_amount"))
)

merchant_fraud_amount.write.mode("overwrite").saveAsTable("`jarvis-catalog`.gold.merchant_fraud_amount")
merchant_fraud_amount.show()

+--------------------+------------------+-----------------------+
|   merchant_category|total_fraud_amount|fraud_transaction_count|
+--------------------+------------------+-----------------------+
|   Department Stores|225647.19000000006|                   2251|
|        Cruise Lines|185946.77999999997|                    165|
|     Wholesale Clubs|         113827.65|                    991|
|     Discount Stores| 81214.89000000001|                    859|
|      Money Transfer|          66101.52|                    725|
|  Electronics Stores|          61171.38|                    402|
|Furniture, Home F...|          56989.45|                    170|
|Miscellaneous Hom...|          34238.45|                    313|
|Telecommunication...|          33625.04|                    162|
|Family Clothing S...|          30051.56|                    385|
|Drug Stores and P...|29926.590000000004|                    479|
|  Passenger Railways|27243.149999999998|                    238|
|Precious 

In [0]:
daily_fraud_losses = (
    s_trans_df
    .filter(F.col("fraud") == "Yes")
    .groupBy("date")
    .agg(
        F.sum("amount").alias("total_fraud_loss"),
        F.count("*").alias("fraud_transaction_count")
    )
    .orderBy("date")
)

daily_fraud_losses.write.mode("overwrite").saveAsTable("`jarvis-catalog`.gold.daily_fraud_losses")
daily_fraud_losses.show()

+----------+------------------+-----------------------+
|      date|  total_fraud_loss|fraud_transaction_count|
+----------+------------------+-----------------------+
|2010-01-01|              0.19|                      1|
|2010-01-03|             339.0|                      1|
|2010-01-04|             11.64|                      2|
|2010-01-05|              8.76|                      1|
|2010-01-07|-48.45999999999998|                      2|
|2010-01-08|            383.24|                      4|
|2010-01-09|              23.1|                      1|
|2010-01-10|            530.01|                      5|
|2010-01-11|             302.7|                      2|
|2010-01-12|           1014.41|                      1|
|2010-01-13|             -95.7|                      2|
|2010-01-15|             24.71|                      1|
|2010-01-16|           1294.01|                      2|
|2010-01-17|            262.02|                      3|
|2010-01-18|485.28000000000003|                 

How many unique users commit fraudulent transactions per week?

In [0]:
weekly_fraud_users = (
    s_trans_df
    .filter(F.col("fraud") == "Yes")
    .withColumn(
        "week",
        F.date_trunc("week", "date")
    )
    .groupBy("week")
    .agg(
        F.countDistinct("client_id").alias("unique_fraud_users")
    )
    .orderBy("week")
)

weekly_fraud_users.write.mode("overwrite").saveAsTable("`jarvis-catalog`.gold.weekly_fraud_users")
weekly_fraud_users.show()

+-------------------+------------------+
|               week|unique_fraud_users|
+-------------------+------------------+
|2009-12-28 00:00:00|                 1|
|2010-01-04 00:00:00|                 5|
|2010-01-11 00:00:00|                 5|
|2010-01-18 00:00:00|                 7|
|2010-01-25 00:00:00|                17|
|2010-02-01 00:00:00|                16|
|2010-02-08 00:00:00|                12|
|2010-02-15 00:00:00|                13|
|2010-02-22 00:00:00|                13|
|2010-03-01 00:00:00|                10|
|2010-03-08 00:00:00|                 6|
|2010-03-15 00:00:00|                14|
|2010-03-22 00:00:00|                11|
|2010-03-29 00:00:00|                11|
|2010-04-05 00:00:00|                13|
|2010-04-12 00:00:00|                11|
|2010-04-19 00:00:00|                 8|
|2010-04-26 00:00:00|                10|
|2010-05-03 00:00:00|                15|
|2010-05-10 00:00:00|                10|
+-------------------+------------------+
only showing top

Do fraud patterns show seasonal or monthly spikes?

In [0]:
seasonal_spikes = (
    s_trans_df
    .withColumn(
        "month_name",
        F.date_format("date", "MMMM")
    )
    .groupBy("month_name")
    .agg(
        F.count("*").alias("transactions"),
        F.sum(
            F.when(F.col("fraud") == "Yes", 1)
             .otherwise(0)
        ).alias("fraud_transactions")
    )
)

seasonal_spikes.write.mode("overwrite").saveAsTable("`jarvis-catalog`.gold.seasonal_spikes")
seasonal_spikes.show()

+----------+------------+------------------+
|month_name|transactions|fraud_transactions|
+----------+------------+------------------+
|     March|     1145390|              1167|
|  November|     1003488|              1089|
|      June|     1118522|               872|
|   January|     1139155|              1003|
|  December|     1038652|              1201|
|     April|     1106182|              1171|
|       May|     1146194|              1097|
|    August|     1156873|              1328|
|  February|     1031351|              1030|
|   October|     1148638|              1175|
| September|     1117795|              1081|
|      July|     1153675|              1118|
+----------+------------+------------------+



How has user behavior changed before versus after a fraudulent event?

In [0]:
first_fraud = (
    s_trans_df
    .filter(F.col("fraud") == "Yes")
    .groupBy("client_id")
    .agg(
        F.min("date").alias("fraud_date")
    )
)

user_behavior_change = (
    s_trans_df
    .join(first_fraud, "client_id")
    .withColumn(
        "period",
        F.when(
            (F.col("date") >= F.date_sub("fraud_date", 30)) &
            (F.col("date") < F.col("fraud_date")),
            "30 Days Before"
        )
        .when(
            (F.col("date") > F.col("fraud_date")) &
            (F.col("date") <= F.date_add("fraud_date", 30)),
            "30 Days After"
        )
    )
    .groupBy("client_id", "period")
    .agg(
        F.count("*").alias("transaction_count"),
        F.round(
            F.sum("amount"),
            2
        ).alias("total_amount")
    )
    .orderBy("client_id", "period")
)

overall_change = (
    user_behavior_change
    .groupBy("period")
    .agg(
        F.avg("transaction_count").alias("avg_transactions"),
        F.avg("total_amount").alias("avg_total_amount")
    )
)

overall_change = overall_change.filter(F.col("period").isNotNull())

overall_change.write.mode("overwrite").saveAsTable("`jarvis-catalog`.gold.overall_change")
overall_change.show()

+--------------+-----------------+------------------+
|        period| avg_transactions|  avg_total_amount|
+--------------+-----------------+------------------+
| 30 Days After|98.17307692307692| 4453.645560200674|
|30 Days Before|93.40753138075314|4029.9353640167315|
+--------------+-----------------+------------------+



Are fraudulent transactions more common on high-value purchases compared to low-value purchases?

In [0]:
fraud_by_value = (
    s_trans_df
    .withColumn(
        "value_category",
        F.when(F.col("amount") < 100, "< $100")
         .when(F.col("amount") < 500, "$100-$499")
         .when(F.col("amount") < 1000, "$500-$999")
         .otherwise("$1000+")
    )
    .groupBy("value_category")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum(
            F.when(F.col("fraud") == "Yes", 1).otherwise(0)
        ).alias("fraud_transactions")
    )
    .withColumn(
        "fraud_rate",
        F.round(
            F.col("fraud_transactions") /
            F.col("total_transactions") * 100,
            2
        )
    )
).orderBy('value_category')

fraud_by_value.write.mode("overwrite").saveAsTable("`jarvis-catalog`.gold.fraud_by_value")
fraud_by_value.show()

+--------------+------------------+------------------+----------+
|value_category|total_transactions|fraud_transactions|fraud_rate|
+--------------+------------------+------------------+----------+
|     $100-$499|           1516248|              4726|      0.31|
|        $1000+|              9185|               100|      1.09|
|     $500-$999|             34179|               256|      0.75|
|        < $100|          11746303|              8250|      0.07|
+--------------+------------------+------------------+----------+



KPI cards for Dashboard

In [0]:
kpi_cards = spark.sql("""SELECT
    COUNT(*) AS total_transactions,
    SUM(CASE WHEN fraud = 'Yes' THEN 1 ELSE 0 END) AS fraud_transactions,
    ROUND(
        SUM(CASE WHEN fraud='Yes' THEN 1 ELSE 0 END)*100.0/COUNT(*),
        2
    ) AS fraud_rate,
    ROUND(
        SUM(CASE WHEN fraud='Yes' THEN amount ELSE 0 END),
        2
    ) AS total_fraud_loss
FROM `jarvis-catalog`.silver.transactions_data;""")

kpi_cards.write.mode("overwrite").saveAsTable("`jarvis-catalog`.gold.kpi_cards")
kpi_cards.show()

+------------------+------------------+----------+----------------+
|total_transactions|fraud_transactions|fraud_rate|total_fraud_loss|
+------------------+------------------+----------+----------------+
|          13305915|             13332|      0.10|      1469648.78|
+------------------+------------------+----------+----------------+

